In [22]:
import numpy as np
import cv2
import os
from collections import Counter
import heapq
from skimage.metrics import structural_similarity as ssim

Q_LUMINANCE = np.array([
    [16, 11, 10, 16, 24, 40, 51, 61],
    [12, 12, 14, 19, 26, 58, 60, 55],
    [14, 13, 16, 24, 40, 57, 69, 56],
    [14, 17, 22, 29, 51, 87, 80, 62],
    [18, 22, 37, 56, 68, 109, 103, 77],
    [24, 35, 55, 64, 81, 104, 113, 92],
    [49, 64, 78, 87, 103, 121, 120, 101],
    [72, 92, 95, 98, 112, 100, 103, 99]
])

def get_quantization_matrix(quality_factor):
    if quality_factor >= 50:
        scale = (100 - quality_factor) / 50.0
    else:
        scale = 50.0 / quality_factor
    
    if scale == 0:
        scale = 0.01

    scaled_matrix = np.floor(Q_LUMINANCE * scale)
    scaled_matrix[scaled_matrix < 1] = 1
    return scaled_matrix.astype(np.int32)

class HuffmanNode:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(data):
    frequency = Counter(data)
    priority_queue = [HuffmanNode(char, freq) for char, freq in frequency.items()]
    heapq.heapify(priority_queue)

    while len(priority_queue) > 1:
        left = heapq.heappop(priority_queue)
        right = heapq.heappop(priority_queue)
        merged = HuffmanNode(None, left.freq + right.freq)
        merged.left = left
        merged.right = right
        heapq.heappush(priority_queue, merged)

    return priority_queue[0]

def generate_huffman_codes(root, current_code="", codes={}):
    if root is None:
        return

    if root.char is not None:
        codes[root.char] = current_code
        return

    generate_huffman_codes(root.left, current_code + "0", codes)
    generate_huffman_codes(root.right, current_code + "1", codes)
    return codes

def huffman_encode(data):
    if not data:
        return "", {}
    root = build_huffman_tree(data)
    codes = generate_huffman_codes(root, "", {})
    encoded_data = "".join([codes[item] for item in data])
    return encoded_data, codes

def huffman_decode(encoded_data, codes):
    if not encoded_data:
        return []
    reverse_codes = {v: k for k, v in codes.items()}
    decoded_data = []
    current_code = ""
    for bit in encoded_data:
        current_code += bit
        if current_code in reverse_codes:
            decoded_data.append(reverse_codes[current_code])
            current_code = ""
    return decoded_data

def compress_image(image_path, quality_factor=50):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found at {image_path}")
    
    h, w = img.shape
    h_pad = (8 - h % 8) % 8
    w_pad = (8 - w % 8) % 8
    img_padded = np.pad(img, ((0, h_pad), (0, w_pad)), mode='edge')
    
    quant_matrix = get_quantization_matrix(quality_factor)
    
    quantized_coeffs = []
    for i in range(0, img_padded.shape[0], 8):
        for j in range(0, img_padded.shape[1], 8):
            block = img_padded[i:i+8, j:j+8].astype(np.float32)
            block -= 128
            dct_block = cv2.dct(block)
            quantized_block = np.round(dct_block / quant_matrix).astype(np.int32)
            quantized_coeffs.extend(quantized_block.flatten())

    encoded_data, huffman_codes = huffman_encode(quantized_coeffs)
    
    return encoded_data, huffman_codes, img.shape, quant_matrix

def decompress_image(encoded_data, huffman_codes, original_shape, quant_matrix):
    decoded_coeffs = huffman_decode(encoded_data, huffman_codes)
    
    num_blocks_h = (original_shape[0] + 7) // 8
    num_blocks_w = (original_shape[1] + 7) // 8
    reconstructed_img = np.zeros((num_blocks_h * 8, num_blocks_w * 8), dtype=np.float32)

    coeff_idx = 0
    for i in range(num_blocks_h):
        for j in range(num_blocks_w):
            block_coeffs = np.array(decoded_coeffs[coeff_idx : coeff_idx + 64]).reshape(8, 8)
            dequantized_block = (block_coeffs * quant_matrix).astype(np.float32)
            idct_block = cv2.idct(dequantized_block)
            idct_block += 128
            reconstructed_img[i*8:(i+1)*8, j*8:(j+1)*8] = idct_block
            coeff_idx += 64
            
    reconstructed_img = np.clip(reconstructed_img, 0, 255)
    
    h, w = original_shape
    reconstructed_img = reconstructed_img[:h, :w]
    
    return reconstructed_img.astype(np.uint8)

if __name__ == "__main__":
    image_path = "images\img.jpg"
    
    if not os.path.exists(image_path):
        print(f"Error: Image file not found at '{image_path}'")
        print("Please download a test image (like lena512.bmp) and place it in the same directory.")
    else:
        original_img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        original_size_bytes = os.path.getsize(image_path)
        pixels = original_img.shape[0] * original_img.shape[1]
        
        print(f"Original Image: '{image_path}'")
        print(f"Dimensions: {original_img.shape}")
        print(f"Size: {original_size_bytes / 1024:.2f} KB\n")
        
        print("| Quality Factor (Q) | Compression Ratio | Bits Per Pixel (bpp) | PSNR (dB) | SSIM   |")
        print("| :----------------: | :---------------: | :------------------: | :-------: | :----: |")
        
        for qf in [90, 70, 50, 30, 10]:
            compressed_data, codes, shape, q_matrix = compress_image(image_path, quality_factor=qf)
            reconstructed_image = decompress_image(compressed_data, codes, shape, q_matrix)
            
            compressed_size_bits = len(compressed_data)
            table_size_bits = len(codes) * 16 
            total_compressed_size_bytes = (compressed_size_bits + table_size_bits) / 8.0
            compression_ratio = original_size_bytes / total_compressed_size_bytes
            bpp = (total_compressed_size_bytes * 8) / pixels
            psnr = cv2.PSNR(original_img, reconstructed_image)
            ssim_score = ssim(original_img, reconstructed_image, data_range=255)

            print(f"| {qf:<18} | {compression_ratio:>15.2f}:1 | {bpp:>20.2f} | {psnr:>9.2f} | {ssim_score:.4f} |")
            
            output_filename = f"reconstructed_q{qf}.png"
            cv2.imwrite(output_filename, reconstructed_image)
            # print(f"-> Saved reconstructed image as '{output_filename}'")

Original Image: 'images\img.jpg'
Dimensions: (2048, 2048)
Size: 565.11 KB

| Quality Factor (Q) | Compression Ratio | Bits Per Pixel (bpp) | PSNR (dB) | SSIM   |
| :----------------: | :---------------: | :------------------: | :-------: | :----: |
| 90                 |            0.59:1 |                 1.86 |     49.88 | 0.9965 |
| 70                 |            0.65:1 |                 1.69 |     47.35 | 0.9946 |
| 50                 |            0.74:1 |                 1.48 |     36.09 | 0.9248 |
| 30                 |            0.86:1 |                 1.28 |     35.20 | 0.8915 |
| 10                 |            0.99:1 |                 1.12 |     31.53 | 0.7990 |


In [6]:
import cv2
import numpy as np

def create_comparison_grid():
    image_info = {
        "Original": "images/img.jpg",
        "Q = 90": "reconstructed_q90.png",
        "Q = 70": "reconstructed_q70.png",
        "Q = 50": "reconstructed_q50.png",
        "Q = 30": "reconstructed_q30.png",
        "Q = 10": "reconstructed_q10.png"
    }

    loaded_images = []
    for label, path in image_info.items():
        try:
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

            cv2.rectangle(img_bgr, (0, 0), (img_bgr.shape[1], 400), (0, 0, 0), -1)
            cv2.putText(
                img_bgr,
                label,
                (50, 300),  # move text lower for visibility
                cv2.FONT_HERSHEY_SIMPLEX,
                10,          # larger font scale
                (255, 255, 255),
                20           # thicker stroke
            )
            loaded_images.append(img_bgr)
        except Exception as e:
            print(f"Error loading {label}: {e}")

        except Exception as e:
            print(f"Error loading image '{path}': {e}")
            print("Please make sure all required image files are in the same folder as the script.")
            return
    if len(loaded_images) == 6:
        top_row = np.hstack([loaded_images[0], loaded_images[1], loaded_images[2]])
        bottom_row = np.hstack([loaded_images[3], loaded_images[4], loaded_images[5]])
        
        comparison_grid = np.vstack([top_row, bottom_row])

        output_filename = "comparison_grid.png"
        cv2.imwrite(output_filename, comparison_grid)
        print(f"\nSuccessfully created the grid!")
        print(f"File saved as: {output_filename}")

    else:
        print("Could not create grid because not all 6 images were loaded.")


if __name__ == "__main__":
    create_comparison_grid()


Successfully created the grid!
File saved as: comparison_grid.png
